# U.S. bank screening with a frozen FDIC dataset

Load **4,313 institution financial records for 2026-06-30** with one call, then select a cohort using a stated reporting quarter. This frozen edition includes NC and OI classes; it is not an insured-only or active-only list, a list of banks for sale, or a latest-data claim. After installation all cells run offline.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

try:
    installed = importlib.metadata.version("us-bank-data") == "0.2.0"
except importlib.metadata.PackageNotFoundError:
    installed = False
if not installed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "us-bank-data==0.2.0"])


In [ ]:
from usbankdata import snapshot

data = snapshot(quarter="2026Q2", edition="2026-q2-r1")
assert len(data.rows) == 4313
print(data.dataset_id)
print(data.document["selection"]["scope"])
print(data.document["coverage"]["fields"]["cet1_ratio_pct"])


## Select a population before comparing it

For this exercise select NC institutions in the N, SM, NM, SB, SI and SL classes, with assets **at least $100 million and below $3 billion** at 2026Q2. These filters are an example research choice, not an official FDIC peer group. Money columns are dollars; 12.5 in a ratio column means 12.5%, and ROA/ROE are already annualized.


In [ ]:
cohort = data.select(states=["NC"], bank_classes=["N", "SM", "NM", "SB", "SI", "SL"],
                     min_assets_usd=100_000_000, max_assets_usd=3_000_000_000)
assert cohort.certs
assert cohort.document["selection"]["anchor_report_date"] == "2026-06-30"
print("Selected institutions:", len(cohort.certs))
for row in cohort.rows[:5]:
    print(row["cert"], row["bank_name"], row["assets_usd"], row["cet1_ratio_pct"])


## Derive one funding measure without filling missing data

Use estimated uninsured **domestic** deposits divided by domestic deposits. Keep an absent numerator or nonpositive denominator missing. Do not replace this denominator with total deposits or treat this measure as a risk score. Derived results below are separate from the reported source fields.


In [ ]:
derived = []
for row in cohort.rows:
    numerator, denominator = row["uninsured_deposits_usd"], row["domestic_deposits_usd"]
    share = None if numerator is None or denominator is None or denominator <= 0 else 100 * numerator / denominator
    derived.append({"cert": row["cert"], "uninsured_domestic_share_pct_derived": share})
print(derived[:5])


In [ ]:
from pathlib import Path
from usbankdata import BankCohort

output = Path("outputs/snapshot-screener")
cohort.export(output / "nc-banks.csv")
cohort.export(output / "nc-banks.json")
assert BankCohort.load(output / "nc-banks.json").rows == cohort.rows
print(output / "nc-banks.manifest.json")


## Exercise

Change the state and asset bounds. How does the number of selected institutions change? Count missing CET1 observations before comparing ratios. Keep the source manifest and state the selection anchor in your answer.

Source: FDIC BankFind Suite; BanksOnSale compilation and metadata (CC BY 4.0), provided through US Bank Data. The archived source edition is as retrieved in October 2026, not the original filing vintage. [Definitions and population](https://github.com/anshuman-pandey/us-bank-data/blob/v0.2.0/docs/COHORTS.md).
